In [ ]:
import numpy as np
import pandas as pd
from difflib import SequenceMatcher
from sklearn.preprocessing import normalize

from rapidfuzz import fuzz
from lightgbm import LGBMRanker
from lightgbm import early_stopping, log_evaluation
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from emoji import emoji_count
from sentence_transformers import SentenceTransformer
from catboost import CatBoostRanker, Pool, MetricVisualizer
from copy import deepcopy



# Решение
Следующие фичи были получены из текстовых данных: совпадение названий брендов(все слова на английсокм языке), совбадение размеров/моделей(все слова, содержащие числа), совпадение категорий, совпадение локаций, также похожесть запроса и названия/описания с использованием *TfidfVectorizer* и модели *sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2* с последующим вычисление *cosine similarity*, длина описания, количество эмодзи в описании. Также цена приводилась к нормализованной шкале по каждому запросу.
В качестве модели использовался *CatBoostRanker* с лоссом *YetiRankPairwise*.

In [4]:
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

In [5]:
train_const = pd.read_parquet('train-dset.parquet')
test_const = pd.read_parquet('test-dset-small.parquet')

In [116]:
train_const.head(200).to_csv("first200.csv", index=False)

In [6]:
groups_const = train_const.groupby('query_id', sort=False).size().values

In [7]:
train_sample = train_const.loc[:groups_const[:20000].sum()]
groups_sample = groups_const[:20000]
train_sample.shape

(232164, 14)

In [8]:
def extract_english_brand_tokens(text: str) -> list:
    """
    Извлекает токены, состоящие из латинских букв, 
    возможно содержащие внутренние дефисы или апострофы.
    Отбрасывает токены, начинающиеся/заканчивающиеся на спецсимволы.
    """
    if pd.isna(text):
        return []
    # Разрешаем дефис и апостроф ВНУТРИ слова
    tokens = re.findall(r"\b[a-zA-Z]+(?:[-'][a-zA-Z]+)*\b", str(text))
    return [t.lower() for t in tokens if t.isascii()]
    

In [9]:
def extract_words_with_digits(text):
    """
    Извлекает все слова, содержащие хотя бы одну цифру.
    
    Слово определяется как последовательность букв, цифр, дефисов или апострофов,
    но обязательно включающая хотя бы одну цифру.
    
    Args:
        text (str or NaN): входной текст
    
    Returns:
        list[str]: список слов с цифрами (в нижнем регистре)
    """
    if pd.isna(text):
        return []
    
    # Находим все "слова", состоящие из букв, цифр, дефисов и апострофов
    tokens = re.findall(r"\b[a-zA-Z0-9]+(?:[-'][a-zA-Z0-9]+)*\b", str(text))
    
    # Фильтруем только те, что содержат хотя бы одну цифру
    words_with_digits = [token.lower() for token in tokens if re.search(r'\d', token)]
    
    return words_with_digits

In [10]:
def has_digit_word_match(row):
        query_tokens = set(extract_words_with_digits(row['query_text']))
        title = str(row['item_title']).lower()
        desc = str(row['item_description']).lower()
        for token in query_tokens:
            if token in title or token in desc:
                return 1
        return 0

In [11]:
def has_brand_match(row):
        query_tokens = set(extract_english_brand_tokens(row['query_text']))
        title = str(row['item_title']).lower()
        desc = str(row['item_description']).lower()
        for token in query_tokens:
            if token in title or token in desc:
                return 1
        return 0

In [12]:
# train - test split
train_length = int(len(groups_sample)*0.8)
test_length = len(groups_sample) - train_length
queries_train = groups_sample[:train_length]
queries_test = groups_sample[train_length:len(groups_sample)]
train_sample_train = train_sample[:sum(queries_train)]
train_sample_test = train_sample[sum(queries_train):sum(groups_sample)]

In [13]:
display(queries_train.shape)
display(queries_test.shape)
display(groups_sample.shape)

(16000,)

(4000,)

(20000,)

In [14]:
train_sample_train.columns

Index(['query_id', 'item_id', 'query_text', 'item_title', 'item_description',
       'query_cat', 'query_mcat', 'query_loc', 'item_cat_id', 'item_mcat_id',
       'item_loc', 'price', 'item_query_click_conv', 'item_contact'],
      dtype='str')

In [15]:
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=5000)
all_text = pd.concat([train_sample_train['query_text'], train_sample_train['item_title'], train_sample_train['item_description']])
%time tfidf.fit(all_text.fillna('').astype(str))

CPU times: user 1min 6s, sys: 806 ms, total: 1min 6s
Wall time: 1min 7s


,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'char_wb'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.",

In [39]:
sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [151]:
sentence_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
def build_features(df):
    df = df.copy()
    
    # Заполним пропуски
    print('Imputing...')
    df['item_description'] = df['item_description'].fillna('')
    df['item_title'] = df['item_title'].fillna('')
    df['query_text'] = df['query_text'].fillna('')
    
    # Совпадение категорий и локаций
    print('cat collisions...')
    df['cat_match'] = (df['query_cat'] == df['item_cat_id']).astype(int)
    df['mcat_match'] = (df['query_mcat'] == df['item_mcat_id']).astype(int)
    df['loc_match'] = (df['query_loc'] == df['item_loc']).astype(int)
    
    # Цена
    print('price transfrom...')
    df['price_log'] = np.log1p(df['price'])
    price_stats = df.groupby('query_id')['price'].agg(['mean', 'std']).reset_index()
    price_stats.columns = ['query_id', 'price_mean', 'price_std']
    df = df.merge(price_stats, on='query_id', how='left')
    df['price_std'] = df['price_std'].fillna(0)
    df['price_norm'] = (df['price'] - df['price_mean']) / (df['price_std'] + 1e-6)

    # Совпадение названий брендов и моделей/размеров
    print('brand collisions...')
    df['brand_match'] = df.apply(has_brand_match, axis=1)
    df['digit_word_match'] = df.apply(has_digit_word_match, axis=1)

    # string distance
    print('string distance...')
    df['query_text'] = df['query_text'].fillna('').astype(str)
    df['item_title'] = df['item_title'].fillna('').astype(str)

    df['matching_ratio'] = df.apply(lambda x:fuzz.ratio(x.query_text.lower(), x.item_title.lower()), axis=1).to_list()

    # TF-IDF similarity
    print('tfidf...')

    %time query_tfidf = tfidf.transform(df['query_text'].fillna('').astype(str))
    %time title_tfidf = tfidf.transform(df['item_title'].fillna('').astype(str))
    %time desc_tfidf = tfidf.transform(df['item_description'].fillna('').astype(str))
    query_tfidf = normalize(query_tfidf, norm='l2', axis=1)
    title_tfidf = normalize(title_tfidf, norm='l2', axis=1)
    desc_tfidf = normalize(desc_tfidf, norm='l2', axis=1)
    # Поэлементное скалярное произведение = cosine similarity
    df['tfidf_title_sim'] = query_tfidf.multiply(title_tfidf).sum(axis=1).A1
    df['tfidf_desc_sim'] = query_tfidf.multiply(desc_tfidf).sum(axis=1).A1

    # sentence embedding
    print('sentence embedding...')

    %time query_embed = sentence_model.encode(np.array(df['query_text'].fillna('').astype(str)))
    %time title_embed = sentence_model.encode(np.array(df['item_title'].fillna('').astype(str)))
    %time desc_embed = sentence_model.encode(np.array(df['item_description'].fillna('').astype(str)))
    
    query_embed = normalize(query_embed, norm='l2', axis=1)
    title_embed = normalize(title_embed, norm='l2', axis=1)
    desc_embed = normalize(desc_embed, norm='l2', axis=1)

    df['embed_title_sim'] = np.sum(query_embed * title_embed, axis=1)
    df['embed_desc_sim'] = np.sum(query_embed * desc_embed, axis=1)

    return df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
test_trasform = build_features(train_sample_test)
train_trasform = build_features(train_sample_train)

Imputing...
cat collisions...
price transfrom...
brand collisions...
string distance...
tfidf...
CPU times: user 2.36 s, sys: 929 μs, total: 2.36 s
Wall time: 2.37 s
CPU times: user 4.25 s, sys: 7.13 ms, total: 4.25 s
Wall time: 4.27 s
CPU times: user 1min 3s, sys: 71.4 ms, total: 1min 3s
Wall time: 1min 3s
sentence embedding...
CPU times: user 38.3 s, sys: 4.94 s, total: 43.2 s
Wall time: 29.4 s
CPU times: user 44.3 s, sys: 5.35 s, total: 49.7 s
Wall time: 32.7 s
CPU times: user 3min 2s, sys: 7.04 s, total: 3min 9s
Wall time: 1min 53s


In [103]:
def count_emojis(text):
    if not isinstance(text, str):
        return 0
    return emoji_count(text)

def add_description_features(df, desc_col='item_description'):
    """
    Добавляет признаки:
      - desc_len: длина описания (в символах)
      - desc_emoji_count: количество эмодзи
    """
    df = df.copy()
    # Заполняем пропуски пустой строкой
    desc_series = df[desc_col].fillna('').astype(str)
    
    # Длина описания
    df['desc_len'] = desc_series.str.len()
    
    # Количество эмодзи
    df['desc_emoji_count'] = desc_series.apply(count_emojis)
    
    return df

In [104]:
train_trasform = add_description_features(train_trasform)
test_trasform = add_description_features(test_trasform)

In [105]:
display(test_trasform)

,query_id,item_id,query_text,item_title,item_description,query_cat,query_mcat,query_loc,item_cat_id,item_mcat_id,...,brand_match,digit_word_match,matching_ratio,tfidf_title_sim,tfidf_desc_sim,embed_title_sim,embed_desc_sim,baseline_score,desc_len,desc_emoji_count
0,78228,2861908857,бонусы в спортмастер,Бонусы Спортмастер бесплатно,"Бесплатные бонусы, без комиссии.\nПодробнее см...",33.0,295.0,659300.0,33,295,...,0,0,75.000000,0.750171,0.269107,0.990845,0.637144,0.750171,671,0
1,78228,7281973097,бонусы в спортмастер,Отдам бонусы и баллы спортмастер,НЕ ПРИСЫЛАТЬ ССЫЛКИ !\n\nБолее 1000 отзывов\nП...,33.0,295.0,659300.0,33,295,...,0,0,73.076923,0.756834,0.572076,0.981116,0.682890,0.756834,359,0
2,78228,3422906745,бонусы в спортмастер,Бонусы спортмастер (7620),📍В наличии огромное количество бонусов Спортма...,33.0,295.0,659300.0,33,295,...,0,0,80.000000,0.950802,0.259769,0.967587,0.611589,0.950802,243,5
3,78228,4372122872,бонусы в спортмастер,Бонусы спортмастер бесплатно,бонусы Спортмастер \n\nчерез заказ: пишите арт...,33.0,295.0,659300.0,33,295,...,0,0,75.000000,0.750171,0.203656,0.990844,0.749393,0.750171,432,0
4,78228,7260898366,бонусы в спортмастер,Бонусы купоны спортмастер 100000,Ссылки не присылать ‼️\n\nСделаю скидку в мага...,33.0,295.0,659300.0,33,295,...,0,0,73.076923,0.651558,0.571554,0.981791,0.487753,0.651558,563,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46707,97958,4387241431,разнорабочий на завод,Рабочий на склад деревообработки,На время обучения минимальная з/п 90000 рублей...,111.0,61.0,653240.0,111,2278193,...,0,0,52.830189,0.389188,0.172469,0.714294,0.453529,0.389188,1000,5
46708,97958,3392688754,разнорабочий на завод,Разнорабочий на производство металлоконструкций,2/2 с 8 до 20-00. Работа с ленточнопильным ста...,111.0,61.0,653240.0,111,2278193,...,0,0,58.823529,0.475986,0.180590,0.552149,0.423640,0.475986,121,0
46709,97958,7255998277,разнорабочий на завод,Рабочий на оконное и стеклопакетное производство,В связи с запуском нового оконного завода и вы...,111.0,61.0,653240.0,111,2278193,...,0,0,43.478261,0.280687,0.247805,0.531005,0.383590,0.280687,819,0
46710,97958,7534643104,разнорабочий на завод,Требуются разнорабочие строительную организацию,Ищем ответственных и трудолюбивых сотрудников ...,111.0,61.0,653240.0,111,2278193,...,0,0,47.058824,0.372120,0.532311,0.555670,0.363214,0.372120,1000,6


In [107]:
# Пример: фильтрация train_sample_train
group_sizes = train_trasform.groupby('query_id').size()
valid_queries = group_sizes[group_sizes > 1].index
train_filtered = train_trasform[train_trasform['query_id'].isin(valid_queries)]

display(train_filtered.head())

# Оставить только query_id, где есть хотя бы один item_contact == 1
valid_queries = train_filtered.groupby('query_id')['item_contact'].sum()
valid_queries = valid_queries[valid_queries > 0].index

train_filtered = train_filtered[train_filtered['query_id'].isin(valid_queries)]




test_filtered = test_trasform.copy()

# Те же правила: >1 товар + хотя бы 1 релевантный
group_sizes = test_filtered.groupby('query_id').size()
valid_queries = group_sizes[group_sizes > 1].index
test_filtered = test_filtered[test_filtered['query_id'].isin(valid_queries)]

has_relevant = test_filtered.groupby('query_id')['item_contact'].sum() > 0
valid_queries = has_relevant[has_relevant].index
test_filtered = test_filtered[test_filtered['query_id'].isin(valid_queries)]

test_filtered = test_filtered.sort_values('query_id').reset_index(drop=True)
queries_test = test_filtered.groupby('query_id').size().values


,query_id,item_id,query_text,item_title,item_description,query_cat,query_mcat,query_loc,item_cat_id,item_mcat_id,...,price_norm,brand_match,digit_word_match,matching_ratio,tfidf_title_sim,tfidf_desc_sim,embed_title_sim,embed_desc_sim,desc_len,desc_emoji_count
0,4,7349717282,ботинки детские zara 21,Ботинки детские Zara,Новые полуботинки фирмы Zara. \nразмеры 21 сте...,29.0,38.0,624480.0,29,2179540,...,-0.319819,1,1,93.023256,0.944001,0.134981,0.786758,0.860331,138,0
1,4,7519735286,ботинки детские zara 21,Детские ботинки Zara унисекс,"Крутые ботинки, в отличном состоянии",29.0,38.0,624480.0,29,2179540,...,-0.799164,1,0,50.980392,0.786779,0.228020,0.865856,0.734297,36,0
2,4,4384449104,ботинки детские zara 21,Ботинки детские zara,Челси димесезонные Zara \nВ идеальном состояни...,29.0,38.0,624480.0,29,2179540,...,1.597561,1,0,93.023256,0.944001,0.135719,0.786758,0.782867,66,0
3,4,7283365509,ботинки детские zara 21,Детские ботиночки Zara 21 размер,АВИТО ДОСТАВКА .21 РАЗМЕР.,29.0,38.0,624480.0,29,2179540,...,-0.856686,1,1,54.545455,0.752552,0.061718,0.959680,0.781620,26,0
4,4,4452768560,ботинки детские zara 21,Детские ботиночки zara размер 21,Детские ботинки Zara \nРазмер 21 - 13 см\nСост...,29.0,38.0,624480.0,29,2179540,...,1.881334,1,1,54.545455,0.752552,0.463345,0.966701,0.795550,121,0


In [108]:
# Пересоздать X_train, y_train, queries_train
feature_cols = ['tfidf_title_sim', 'brand_match', 'digit_word_match', 'cat_match',
       'mcat_match', 'loc_match', 'price_norm', 'tfidf_desc_sim', 'item_query_click_conv',
       'embed_title_sim','embed_desc_sim', 'desc_len','desc_emoji_count']
X_train = train_filtered[feature_cols]
y_train = np.array(train_filtered.item_contact)
X_test = test_filtered[feature_cols]
y_test = test_filtered['item_contact'].values

In [138]:
train = Pool(
    data=X_train,
    label=y_train,
    group_id=train_filtered['query_id'].values
)

test = Pool(
    data=X_test,
    label=y_test,
    group_id=test_filtered['query_id'].values
)

In [140]:
default_parameters = {
    'iterations': 2000,
    'custom_metric': ['NDCG', 'PFound', 'AverageGain:top=10'],
    'verbose': False,
    'random_seed': 0,
}

parameters = {}
def fit_model(loss_function, additional_params=None, train_pool=train, test_pool=test):
    parameters = deepcopy(default_parameters)
    parameters['loss_function'] = loss_function
    parameters['train_dir'] = loss_function

    if additional_params is not None:
        parameters.update(additional_params)

    model = CatBoostRanker(**parameters)
    model.fit(train_pool, eval_set=test_pool, plot=True)

    return model

In [146]:
model =fit_model('YetiRankPairwise')


MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [152]:
test_full_transform = build_features(test_const)

Imputing...
cat collisions...
price transfrom...
brand collisions...
string distance...
tfidf...
CPU times: user 4.52 s, sys: 8.35 ms, total: 4.53 s
Wall time: 4.57 s
CPU times: user 8.95 s, sys: 14.3 ms, total: 8.96 s
Wall time: 9.03 s
CPU times: user 2min 27s, sys: 481 ms, total: 2min 28s
Wall time: 2min 29s
sentence embedding...
CPU times: user 1min 21s, sys: 12.2 s, total: 1min 33s
Wall time: 1min 4s
CPU times: user 1min 34s, sys: 13.6 s, total: 1min 47s
Wall time: 1min 11s
CPU times: user 6min 10s, sys: 18.4 s, total: 6min 29s
Wall time: 4min 1s


In [155]:
test_full_transform = add_description_features(test_full_transform)

In [158]:
pred = model.predict(test_full_transform[feature_cols])

In [160]:
test_full_transform

,query_id,item_id,query_text,item_title,item_description,query_cat,query_mcat,query_loc,item_cat_id,item_mcat_id,...,price_norm,brand_match,digit_word_match,matching_ratio,tfidf_title_sim,tfidf_desc_sim,embed_title_sim,embed_desc_sim,desc_len,desc_emoji_count
0,55,7540855789,1 сентября,Воздушные и гелиевые шары на 1 сентября,ВОЗДУШНЫЕ ШАРЫ К 1 СЕНТЯБРЯ 🍂\n\n🎒 Друзья! Мы ...,114.0,63.0,637640.0,114,2301564,...,-0.240760,0,1,40.816327,0.453113,0.119774,0.942641,0.716723,1000,8
1,55,7506720336,1 сентября,1 сентября фотозона,🎈Фотозона из шаров на 1 сентября – создайте пр...,114.0,63.0,637640.0,114,2301564,...,-0.125173,0,1,68.965517,0.665618,0.255996,0.984290,0.702949,621,12
2,55,3110733862,1 сентября,Букет на 1 сентября из зефира,Букеты на 1 сентября \n\nФото 1. Букет Пионов ...,114.0,63.0,637640.0,114,1090077,...,-0.215179,0,1,51.282051,0.540186,0.089376,0.936044,0.611329,1000,0
3,55,7587733901,1 сентября,Спектакль-пантомима на 1 сентября,Спектакль-пантомима на 1 сентября!\n\n👍🏼Идеаль...,114.0,63.0,637640.0,114,2301563,...,-0.243602,0,1,46.511628,0.546240,0.278416,0.929429,0.658981,746,12
4,55,7552455685,1 сентября,Воздушные гелиевые шары с доставкой 1 сентября,Воздушные шары для оформления школ и классов н...,114.0,63.0,637640.0,114,2301564,...,-0.241233,0,1,35.714286,0.402302,0.272154,0.938762,0.752693,492,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
335343,824759,3584540577,утилизация бытовой техники,"Вывоз оргтехники, сотовых телефонов, телевизоров","Скупка на утилизацию не рабочей оргтехники, ко...",114.0,2094222.0,662210.0,114,2301618,...,-0.175633,0,0,32.432432,0.228438,0.400178,0.883591,0.665316,432,0
335344,824759,4130130433,утилизация бытовой техники,Прием компьютерных и радиоэлектроных плат,Дopoгo пpинимaeм электрoнные плaты и кoмпьютеp...,114.0,2094222.0,662210.0,114,2301618,...,-0.175576,0,0,23.880597,0.008505,0.046395,0.859523,0.663768,1000,0
335345,824759,2303947188,утилизация бытовой техники,Вывоз мусора. Демонтаж. Вывоз строительного му...,🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩\r\nВывоз мусора\r\nВывоз СТРО...,114.0,2094222.0,662210.0,114,2301617,...,-0.175633,0,0,28.947368,0.010417,0.104969,0.963420,0.721888,1000,86
335346,824759,1864686952,утилизация бытовой техники,Вывоз мусора,Вывоз мусора контейнерами 8 кубов\n Работаем в...,114.0,2094222.0,662210.0,114,2301617,...,-0.175633,0,0,21.052632,0.008517,0.191239,0.996231,0.753759,1000,4


In [166]:
result = test_full_transform[['query_id', 'item_id']]
result['score'] = pred
result = result.sort_values(by=['query_id', 'score'], ascending=[True, False])
result = result[['query_id', 'item_id']]


In [168]:
result.to_csv(
    'solution.csv', 
    header=['query_id', 'item_id'], 
    index=False,
)